# Sales Data Analysis and Business Insights
**Student:** Prathamesh Jadhav  
**Project:** AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026  
**Dataset:** Tableau Sample Superstore (fictional educational retail data)

## Project overview
This notebook performs data validation, cleaning, exploratory data analysis (EDA), business KPI calculation, visualisation, and simple trend analysis. It does not claim to use a trained AI model; it is an intermediate data analytics project with a clearly labelled optional forecasting extension.

## Dataset setup
Download the **Superstore Sales** sample from [Tableau Public Sample Data](https://public.tableau.com/app/learn/sample-data) or use the alternative dataset page at [Kaggle – Superstore Dataset](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final). Save the order-level CSV in the same folder as this notebook and name it `Sample - Superstore.csv` or `Sample_Superstore.csv`.

The Tableau sample may be provided as Excel rather than CSV. If so, save/export the Orders sheet as CSV. Do not include the People or Returns sheets in the orders CSV.

## Important integrity note
All metrics and charts below are computed from the dataset at runtime. Do not report numerical findings until you run the notebook successfully against the actual downloaded file. Dataset copies can differ, so the notebook prints row counts, dates, and KPI values from the file used.

## 1. Setup and imports
Run this notebook from top to bottom. If the dataset is missing, the notebook explains where to obtain it.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python data analytics environment ready.")
print("Output directory:", OUTPUT_DIR.resolve())

## 2. Load the dataset
The loader checks common filenames. If your file has a different name, update `DATA_CANDIDATES` below.

In [ ]:
DATA_CANDIDATES = [
    Path("Sample - Superstore.csv"),
    Path("Sample_Superstore.csv"),
    Path("Sample - Superstore.xlsx"),
    Path("Sample_Superstore.xlsx"),
    Path("superstore.csv"),
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Download Tableau's Superstore Sales sample from "
        "https://public.tableau.com/app/learn/sample-data, export the Orders sheet "
        "as CSV, and save it beside this notebook as 'Sample - Superstore.csv'. "
        "A Kaggle alternative is https://www.kaggle.com/datasets/vivek468/superstore-dataset-final"
    )

if data_path.suffix.lower() in {".xlsx", ".xls"}:
    df_raw = pd.read_excel(data_path)
else:
    df_raw = pd.read_csv(data_path, encoding_errors="replace")

print(f"Loaded: {data_path}")
print(f"Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
display(df_raw.head())

## 3. Clean and validate
The analysis standardises column names, parses dates, checks missing values and duplicates, and creates month, year, shipping-days, and margin fields where possible.

In [ ]:
def normalise_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df = df_raw.copy()
df.columns = [normalise_column_name(c) for c in df.columns]

# Map common field names from the Sample Superstore dataset.
aliases = {
    "order_date": ["order_date", "orderdate"],
    "ship_date": ["ship_date", "shipdate"],
    "order_id": ["order_id", "orderid"],
    "sales": ["sales", "revenue"],
    "profit": ["profit"],
    "quantity": ["quantity", "qty"],
    "discount": ["discount"],
    "category": ["category"],
    "sub_category": ["sub_category", "subcategory"],
    "region": ["region"],
    "segment": ["segment"],
    "product_name": ["product_name", "productname"],
    "customer_id": ["customer_id", "customerid"],
}
rename_map = {}
for canonical, options in aliases.items():
    for option in options:
        if option in df.columns:
            rename_map[option] = canonical
            break
df = df.rename(columns=rename_map)

required = ["sales", "profit"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(
        f"Required column(s) missing: {missing_required}. Found columns: {list(df.columns)}. "
        "Please use the Orders sheet from Sample Superstore."
    )

for col in ["sales", "profit", "quantity", "discount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["order_date", "ship_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

quality_before = {
    "rows_before_cleaning": int(len(df)),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_cells": int(df.isna().sum().sum()),
}
df = df.drop_duplicates().copy()

if "order_date" in df.columns:
    df["order_year"] = df["order_date"].dt.year
    df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
if {"ship_date", "order_date"}.issubset(df.columns):
    df["days_to_ship"] = (df["ship_date"] - df["order_date"]).dt.days
if {"profit", "sales"}.issubset(df.columns):
    df["profit_margin"] = np.where(df["sales"] != 0, df["profit"] / df["sales"], np.nan)
if "discount" in df.columns:
    df["discount_band"] = pd.cut(
        df["discount"],
        bins=[-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
        labels=["0%", ">0–10%", ">10–20%", ">20–30%", ">30–50%", ">50%"],
        include_lowest=True,
    )

quality_after = {
    "rows_after_duplicate_removal": int(len(df)),
    "missing_cells_after_cleaning": int(df.isna().sum().sum()),
}
print("Data quality summary")
display(pd.DataFrame([quality_before | quality_after]).T.rename(columns={0: "value"}))
print("Columns available:", ", ".join(df.columns))

## 4. Key performance indicators (KPIs)
Sales, profit, profit margin, quantity, order count, and average sales per order are calculated from the loaded dataset.

In [ ]:
sales_total = df["sales"].sum(skipna=True)
profit_total = df["profit"].sum(skipna=True)
order_count = df["order_id"].nunique() if "order_id" in df.columns else len(df)
quantity_total = df["quantity"].sum(skipna=True) if "quantity" in df.columns else np.nan

kpis = pd.DataFrame({
    "Metric": ["Total sales", "Total profit", "Profit margin", "Distinct orders", "Total quantity", "Average sales per order"],
    "Value": [
        sales_total,
        profit_total,
        profit_total / sales_total if sales_total else np.nan,
        order_count,
        quantity_total,
        sales_total / order_count if order_count else np.nan,
    ],
})
display(kpis.style.format({"Value": lambda x: f"{x:,.2f}" if pd.notna(x) else "N/A"}))
kpis.to_csv(OUTPUT_DIR / "kpi_summary.csv", index=False)

print("These values are computed from the selected dataset; use the actual output in your report.")

## 5. Exploratory analysis and visualisations
The charts compare performance by category, region, month, and discount band. Negative profit is retained because it is a meaningful business signal.

In [ ]:
def save_bar_chart(data, x, y, title, filename, rotation=30):
    if data.empty:
        print(f"Skipping {title}: no data available.")
        return
    ax = data.plot(kind="bar", x=x, y=y, legend=False, figsize=(10, 5))
    ax.set_title(title)
    ax.set_xlabel(x.replace("_", " ").title())
    ax.set_ylabel(y.replace("_", " ").title())
    ax.tick_params(axis="x", labelrotation=rotation)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

if "category" in df.columns:
    category_summary = df.groupby("category", dropna=False).agg(
        sales=("sales", "sum"), profit=("profit", "sum")
    ).reset_index().sort_values("sales", ascending=False)
    display(category_summary)
    category_summary.to_csv(OUTPUT_DIR / "category_summary.csv", index=False)
    save_bar_chart(category_summary, "category", "sales", "Sales by Category", "sales_by_category.png")
    save_bar_chart(category_summary.sort_values("profit"), "category", "profit", "Profit by Category", "profit_by_category.png")

if "region" in df.columns:
    region_summary = df.groupby("region", dropna=False).agg(
        sales=("sales", "sum"), profit=("profit", "sum")
    ).reset_index().sort_values("sales", ascending=False)
    display(region_summary)
    region_summary.to_csv(OUTPUT_DIR / "region_summary.csv", index=False)
    save_bar_chart(region_summary, "region", "sales", "Sales by Region", "sales_by_region.png")

if "sub_category" in df.columns:
    subcat_summary = df.groupby("sub_category", dropna=False).agg(
        sales=("sales", "sum"), profit=("profit", "sum")
    ).reset_index().sort_values("profit")
    display(subcat_summary)
    subcat_summary.to_csv(OUTPUT_DIR / "subcategory_summary.csv", index=False)
    save_bar_chart(subcat_summary, "sub_category", "profit", "Profit by Sub-Category", "profit_by_subcategory.png", rotation=60)

## 6. Business questions
Use the tables and charts to answer: Which categories and regions generate sales and profit? Are any sub-categories loss-making? How do sales change over time? Is discounting associated with weaker profit margins?

In [ ]:
if "order_month" in df.columns:
    monthly = df.dropna(subset=["order_month"]).groupby("order_month", as_index=False).agg(
        sales=("sales", "sum"), profit=("profit", "sum")
    ).sort_values("order_month")
    display(monthly.head())
    monthly.to_csv(OUTPUT_DIR / "monthly_trend.csv", index=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(monthly["order_month"], monthly["sales"], marker="o", label="Sales")
    ax.plot(monthly["order_month"], monthly["profit"], marker=".", label="Profit")
    ax.set_title("Monthly Sales and Profit Trend")
    ax.set_xlabel("Order month")
    ax.set_ylabel("Amount")
    ax.tick_params(axis="x", rotation=75)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "monthly_sales_profit_trend.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Order date column unavailable; monthly trend not generated.")

## 7. Export analysis outputs
CSV summary tables and charts are saved to an `outputs/` folder for inclusion in the project report.

In [ ]:
if "discount_band" in df.columns:
    discount_summary = df.groupby("discount_band", observed=False, dropna=False).agg(
        rows=("profit", "size"),
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        average_profit=("profit", "mean"),
    ).reset_index()
    discount_summary["profit_margin"] = np.where(
        discount_summary["sales"] != 0,
        discount_summary["profit"] / discount_summary["sales"],
        np.nan,
    )
    display(discount_summary)
    discount_summary.to_csv(OUTPUT_DIR / "discount_analysis.csv", index=False)

    plot_data = discount_summary.dropna(subset=["profit_margin"])
    if not plot_data.empty:
        ax = plot_data.plot(kind="bar", x="discount_band", y="profit_margin", legend=False, figsize=(10, 5))
        ax.set_title("Profit Margin by Discount Band")
        ax.set_xlabel("Discount band")
        ax.set_ylabel("Profit margin (ratio)")
        ax.tick_params(axis="x", rotation=30)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "profit_margin_by_discount.png", dpi=160, bbox_inches="tight")
        plt.show()
else:
    print("Discount column unavailable; discount analysis not generated.")

## 8. Conclusions and limitations
The final business recommendations should be written after inspecting the generated outputs. This is fictional sample data and should not be treated as evidence about a real company. Association between discounts and profit is not proof of causation.

In [ ]:
# Automated, data-dependent observations (not causal claims)
print(f"Rows analysed after duplicate removal: {len(df):,}")
print(f"Total sales: {sales_total:,.2f}")
print(f"Total profit: {profit_total:,.2f}")
if "category" in df.columns:
    best_sales = category_summary.sort_values("sales", ascending=False).iloc[0]
    best_profit = category_summary.sort_values("profit", ascending=False).iloc[0]
    worst_profit = category_summary.sort_values("profit", ascending=True).iloc[0]
    print(f"Highest-sales category: {best_sales['category']} ({best_sales['sales']:,.2f} sales)")
    print(f"Highest-profit category: {best_profit['category']} ({best_profit['profit']:,.2f} profit)")
    print(f"Lowest-profit category: {worst_profit['category']} ({worst_profit['profit']:,.2f} profit)")
if "region" in df.columns:
    best_region = region_summary.sort_values("profit", ascending=False).iloc[0]
    print(f"Highest-profit region: {best_region['region']} ({best_region['profit']:,.2f} profit)")
print("\nInterpret findings with context. A low-profit group may differ in size, product mix, discounting, or operating costs.")
print("Do not interpret a relationship between discount and profit as proof that discounts caused the outcome.")

In [ ]:
# Optional extension: a simple time-based baseline forecast
# This is a demonstration, not a production forecast. It needs enough monthly history.
if "order_month" in df.columns:
    monthly_forecast = df.dropna(subset=["order_month"]).groupby("order_month")["sales"].sum().sort_index()
    if len(monthly_forecast) >= 6:
        baseline = monthly_forecast.rolling(window=3, min_periods=3).mean().shift(1)
        forecast_check = pd.DataFrame({"actual_sales": monthly_forecast, "three_month_moving_average_baseline": baseline})
        display(forecast_check.tail(12))
        forecast_check.to_csv(OUTPUT_DIR / "optional_moving_average_baseline.csv")
        print("The moving-average baseline is illustrative. Evaluate forecast errors on a chronological holdout before using it for decisions.")
    else:
        print("Not enough monthly observations for the optional baseline.")
else:
    print("Optional baseline skipped because order dates are unavailable.")

In [ ]:
## Exported deliverables
The `outputs/` directory contains KPI and grouped summary CSV files plus charts generated from the current dataset. Use these actual outputs when writing final numerical conclusions.

## Final report checklist
- [ ] Confirm dataset source and exact file used.
- [ ] Record row count and date range printed by the notebook.
- [ ] Insert actual KPI values and charts from `outputs/`.
- [ ] Explain at least two limitations.
- [ ] Do not describe the project as an AI model unless you implement and evaluate one.
- [ ] Keep the report consistent with the notebook output.